# Assignment: Natural Language Inference with BERT and LLM Comparison

## Overview

In this assignment, you are expected to design and implement a small experimental Natural Language Processing (NLP) pipeline using the Python programming language. Unlike strictly guided projects, this assignment allows flexibility in your experimental design and implementation choices.

You will use the **SNLI (Stanford Natural Language Inference)** dataset available on Hugging Face:

https://huggingface.co/datasets/stanfordnlp/snli

The dataset consists of sentence pairs:

* **Premise**
* **Hypothesis**

Your task is to determine the relationship between these two sentences using one of the following labels:

* **Entailment:** The hypothesis can logically be inferred from the premise.
* **Contradiction:** The hypothesis contradicts the premise.
* **Neutral:** The hypothesis is neither supported nor contradicted by the premise.

The main objective is to:

1. Fine-tune a BERT-based model using the SNLI training set.
2. Use a ready-made Large Language Model (LLM) for inference.
3. Compare the performances and outputs of both approaches.

## Task 1: Data Exploration, Analysis and Preprocessing

In this stage, you should prepare the dataset using the Python `datasets` library. You are expected to explore and understand the dataset through analytical outputs and visualizations generated with code.

Possible analyses include (but are not limited to):

* Sentence length distributions
* Total word counts
* Stopword ratios
* Label distributions
* Vocabulary statistics
* Premise vs. hypothesis length comparisons
* Class imbalance analysis

You may include tables, plots, or statistical summaries to support your observations.

### Preprocessing

You are free to apply any preprocessing steps you consider useful, such as:

* Lowercasing
* Removing special characters
* Stopword removal
* Token normalization
* Truncation or padding strategies
* Filtering invalid samples

At the end of this stage, ensure that the whole dataset including validation and test splits is fully prepared for training phase.

> **Note:** This stage is intentionally flexible. If you choose not to perform any preprocessing or exploratory analysis, you must provide a reasonable explanation in markdown format discussing why you skipped these steps.

---
### Task 1: Your Work Starts Here

### Setup

Runs on Kaggle or Colab (one GPU). All outputs (metrics, predictions, figures, raw LLM answers) are written to `OUT_DIR`.
Set the environment variable `PA5_SMOKE=1` for a quick end-to-end test on tiny subsets.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # one GPU: deterministic batch size, Ollama shares the same GPU
import sys, json, time, random, re, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_dataset
from IPython.display import display

SMOKE = os.environ.get("PA5_SMOKE") == "1"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("pa5_outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
LABELS = ["Entailment", "Neutral", "Contradiction"]
N_COMPARE = 30 if SMOKE else 1000          # first N test samples used for the BERT vs. LLM comparison

def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name), dpi=130, bbox_inches="tight")

print(f"Device: {DEVICE} ({GPU_NAME}) | torch {torch.__version__} | smoke test: {SMOKE}")
print("Outputs ->", OUT_DIR)

### Loading and cleaning the data

Samples labelled `-1` (no annotator consensus) are removed from every split.

In [ ]:
raw = load_dataset("stanfordnlp/snli")
ds = raw.filter(lambda x: x["label"] != -1)

sizes = pd.DataFrame({s: {"original": len(raw[s]), "after removing -1": len(ds[s])} for s in raw}).T
display(sizes)

label_dist = pd.DataFrame({
    s: pd.Series(ds[s]["label"]).map(dict(enumerate(LABELS))).value_counts(normalize=True).reindex(LABELS)
    for s in ds
}).round(3)
print("Label distribution (share of each split):")
display(label_dist)

In [ ]:
# Training split as a DataFrame for the exploratory plots
df = ds["train"].to_pandas()
label_map = {0: 'Entailment', 1: 'Neutral', 2: 'Contradiction'}
df['label_name'] = df['label'].map(label_map)
df['premise'] = df['premise'].fillna('')
df['hypothesis'] = df['hypothesis'].fillna('')
df['premise_len'] = df['premise'].str.split().str.len()
df['hypothesis_len'] = df['hypothesis'].str.split().str.len()

# PLOT 1: label distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='label_name', hue='label_name', order=LABELS, palette='viridis', legend=False)
plt.title("Label Distribution in SNLI Training Set")
plt.xlabel("Class Label"); plt.ylabel("Number of Sentence Pairs")
savefig("t1_label_distribution.png"); plt.show()

# PLOT 2: sentence lengths per class
df_melted = df.melt(id_vars=['label_name'], value_vars=['premise_len', 'hypothesis_len'],
                    var_name='sentence_type', value_name='length')
df_melted['sentence_type'] = df_melted['sentence_type'].map({'premise_len': 'Premise', 'hypothesis_len': 'Hypothesis'})
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_melted, x='label_name', y='length', hue='sentence_type', palette='Set2', showfliers=False)
plt.title("Sentence Length Distribution per Class")
plt.xlabel("Class Label"); plt.ylabel("Number of Words"); plt.legend(title="Sentence Type")
savefig("t1_lengths_per_class.png"); plt.show()

# PLOT 3: premise vs. hypothesis length (2D density)
plt.figure(figsize=(9, 6))
sns.histplot(data=df, x='premise_len', y='hypothesis_len', bins=40, pthresh=.01, cmap="mako", cbar=True)
plt.title("Premise vs. Hypothesis Length (2D Density)")
plt.xlabel("Premise Length (words)"); plt.ylabel("Hypothesis Length (words)")
plt.xlim(0, 40); plt.ylim(0, 25)   # ~99% of the data; a few very long outliers would squash the plot
savefig("t1_length_density.png"); plt.show()

print(df[['premise_len', 'hypothesis_len']].describe().round(2))

### Lexical cues per label

Two simple statistics that hint at the shortcuts a model can learn: the share of hypothesis words that also appear in the
premise (word overlap) and the share of hypotheses containing a negation word.

In [ ]:
NEG_RE = re.compile(r"\b(?:not|no|never|nobody|nothing|none|cannot)\b|n't")
WORD_RE = re.compile(r"[a-z]+")

def words(s):
    return set(WORD_RE.findall(s.lower()))

df['overlap'] = [len(words(h) & words(p)) / max(len(words(h)), 1) for p, h in zip(df.premise, df.hypothesis)]
df['hyp_negation'] = df.hypothesis.str.lower().str.contains(NEG_RE)

cues = df.groupby('label_name').agg(
    hypothesis_words=('hypothesis_len', 'mean'),
    word_overlap=('overlap', 'mean'),
    negation_rate=('hyp_negation', 'mean'),
).reindex(LABELS).round(3)
display(cues)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x='label_name', y='overlap', order=LABELS, ax=ax[0], showfliers=False)
ax[0].set_title("Hypothesis-premise word overlap"); ax[0].set_xlabel("")
cues['negation_rate'].plot.bar(ax=ax[1], color="#4c72b0", rot=0)
ax[1].set_title("Share of hypotheses with a negation word"); ax[1].set_xlabel("")
plt.tight_layout(); savefig("t1_lexical_cues.png"); plt.show()

### Preprocessing decisions

- **Removed:** samples with label `-1` in all three splits.
- **Not applied: stopword removal and special-character removal.** Function words such as *not*, *no* and *nobody*
  carry the strongest signal for *Contradiction* (see the negation rate above), so removing them would delete label information.
- **Lowercasing** is handled by the `bert-base-uncased` tokenizer itself.
- **Truncation / padding:** pairs are truncated to 128 WordPiece tokens (SNLI pairs are far shorter, see Task 2) and padded
  dynamically per batch instead of to a fixed length, which avoids wasting computation on padding.

### Task 1: Your Work Ends Here
---

## Task 2: BERT Fine-Tuning

In this stage, you will fine-tune a BERT-based model using **only the training split** of the dataset.

The dataset already contains predefined train, validation, and test splits via the `datasets` library. You may use any transformer-based BERT architecture and any deep learning framework or library you prefer (e.g., Hugging Face Transformers, PyTorch, TensorFlow, etc.).

### Requirements

You must clearly explain and demonstrate:

* How the two texts were combined
* Tokenization strategy
* Input formatting
* Hyperparameter choices
* Training setup
* Validation strategy
* Experimental observations

Include outputs, comments, and explanations throughout your notebook or report (markdown explanations are required).

> **Important:** The test set must **not** be used during training or validation.

### Evaluation and Visualization

There will be three primary evaluation metrics you should consider:

* Confusion Matrix
* Accuracy
* Macro F1-Score

You are expected to use the **entire test split** of the dataset during the evaluation phase.

You must include visual outputs related to the training process, such as:

* Training/validation loss curves
* Confusion Matrix, Accuracy, and Macro F1-Score plots

You are encouraged to interpret and discuss your results.

---
### Task 2: Your Work Starts Here

### Input formatting and tokenization

The premise and hypothesis are passed to the tokenizer as a **sentence pair**:
`[CLS] premise [SEP] hypothesis [SEP]`. Segment ids (`token_type_ids`) are 0 for the premise and 1 for the hypothesis,
so BERT knows which sentence each token belongs to. The `[CLS]` representation feeds a 3-way classification head.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds, val_ds, test_ds = ds["train"], ds["validation"], ds["test"]
if SMOKE:
    train_ds = train_ds.shuffle(seed=SEED).select(range(2000))
    val_ds, test_ds = val_ds.select(range(200)), test_ds.select(range(200))

def tokenize(batch):
    return tokenizer(batch["premise"], batch["hypothesis"], truncation=True, max_length=MAX_LEN)

enc = {name: d.map(tokenize, batched=True, remove_columns=["premise", "hypothesis"]).rename_column("label", "labels")
       for name, d in [("train", train_ds), ("validation", val_ds), ("test", test_ds)]}

lens = pd.Series([len(x) for x in enc["train"].select(range(min(50_000, len(enc["train"]))))["input_ids"]])
print("Pair length in WordPiece tokens (first 50k training pairs):")
print(lens.describe(percentiles=[.5, .95, .99]).round(1))
print("Share longer than", MAX_LEN, ":", float((lens > MAX_LEN).mean()))

ex = enc["train"][0]
print("\nExample input:", tokenizer.decode(ex["input_ids"]))
print("token_type_ids:", ex["token_type_ids"])

### Training setup

- **Model:** `bert-base-uncased` with a 3-class head.
- **Hyperparameters:** AdamW, learning rate 2e-5 with linear decay, weight decay 0.01, batch size 32, **1 epoch** over the full
  training split (549k pairs), mixed precision (fp16) on GPU. One epoch is usually enough on SNLI because the training set
  is very large; more epochs mainly cost time.
- **Validation:** the validation split is evaluated every 2,000 steps (loss, accuracy, macro F1). The **test split is not
  used** until training is finished.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)})

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds), "macro_f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "bert_tmp"),
    num_train_epochs=1,
    max_steps=40 if SMOKE else -1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    eval_strategy="steps",
    eval_steps=20 if SMOKE else 2000,
    logging_steps=5 if SMOKE else 200,
    save_strategy="no",
    fp16=(DEVICE == "cuda"),
    seed=SEED,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=enc["train"], eval_dataset=enc["validation"],
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)

t0 = time.time()
train_result = trainer.train()
train_minutes = (time.time() - t0) / 60
print(f"Training time: {train_minutes:.1f} min on {GPU_NAME}")
json.dump(trainer.state.log_history, open(os.path.join(OUT_DIR, "bert_log_history.json"), "w"), indent=1)

In [ ]:
# Training / validation curves
hist = pd.DataFrame(trainer.state.log_history)
tr = hist.dropna(subset=["loss"]) if "loss" in hist else pd.DataFrame()
ev = hist.dropna(subset=["eval_loss"])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
if len(tr):
    ax[0].plot(tr["step"], tr["loss"], label="training loss", alpha=.8)
ax[0].plot(ev["step"], ev["eval_loss"], "o-", label="validation loss")
ax[0].set_xlabel("step"); ax[0].set_ylabel("cross-entropy"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(ev["step"], ev["eval_accuracy"], "o-", label="validation accuracy")
ax[1].plot(ev["step"], ev["eval_macro_f1"], "s--", label="validation macro F1")
ax[1].set_xlabel("step"); ax[1].set_title("Validation metrics"); ax[1].legend()
plt.tight_layout(); savefig("t2_training_curves.png"); plt.show()
display(ev[["step", "eval_loss", "eval_accuracy", "eval_macro_f1"]].round(4))

### Evaluation on the full test split

In [ ]:
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
pred = trainer.predict(enc["test"])
test_seconds = time.time() - t0
y_true_bert = pred.label_ids
y_pred_bert = pred.predictions.argmax(-1)

bert_test = {
    "n": int(len(y_true_bert)),
    "accuracy": float(accuracy_score(y_true_bert, y_pred_bert)),
    "macro_f1": float(f1_score(y_true_bert, y_pred_bert, average="macro")),
    "batched_ms_per_example": 1000 * test_seconds / len(y_true_bert),
    "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else None,
    "train_minutes": train_minutes,
}
print(json.dumps(bert_test, indent=1))
print(classification_report(y_true_bert, y_pred_bert, target_names=LABELS, digits=4))

cm = confusion_matrix(y_true_bert, y_pred_bert)
fig, ax = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 2]})
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS, ax=ax[0])
ax[0].set_title(f"BERT confusion matrix (full test set, n={len(y_true_bert)})")
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True")
ax[1].bar(["Accuracy", "Macro F1"], [bert_test["accuracy"], bert_test["macro_f1"]], color=["#4c72b0", "#55a868"])
for i, v in enumerate([bert_test["accuracy"], bert_test["macro_f1"]]):
    ax[1].text(i, v + .005, f"{v:.4f}", ha="center")
ax[1].set_ylim(0, 1); ax[1].set_title("BERT test metrics")
plt.tight_layout(); savefig("t2_bert_test.png"); plt.show()

In [ ]:
# Single-example latency (batch size 1), for the cost / latency comparison in Task 5
bert_model = trainer.model.eval()
sample = test_ds.select(range(min(200, len(test_ds))))
times = []
with torch.no_grad():
    for p, h in zip(sample["premise"], sample["hypothesis"]):
        x = tokenizer(p, h, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        t = time.perf_counter(); bert_model(**x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t)
bert_test["single_ms_per_example"] = 1000 * float(np.median(times[5:]))
print(f"Median latency, batch size 1: {bert_test['single_ms_per_example']:.1f} ms")

# Save BERT predictions for the first N_COMPARE test samples (used again in Task 4)
cmp = test_ds.select(range(min(N_COMPARE, len(test_ds))))
compare_df = pd.DataFrame({"idx": range(len(cmp)), "premise": list(cmp["premise"]),
                           "hypothesis": list(cmp["hypothesis"]), "gold": list(cmp["label"]),
                           "bert_pred": y_pred_bert[:len(cmp)]})
compare_df.to_csv(os.path.join(OUT_DIR, "bert_predictions_first1000.csv"), index=False)
json.dump(bert_test, open(os.path.join(OUT_DIR, "bert_test_metrics.json"), "w"), indent=1)

# Free GPU memory for the LLMs
del trainer, model, bert_model
import gc; gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

### Task 2: Your Work Ends Here
---

## Task 3: Evaluating LLMs on NLI

In this task, the objective is to evaluate how powerful general-purpose Large Language Models (LLMs) perform on the Natural Language Inference (NLI) task **without task-specific fine-tuning**, and to compare their behavior against the fine-tuned BERT model developed in Task 1.

Unlike BERT, which produces deterministic class predictions after supervised fine-tuning, LLMs rely entirely on prompting and instruction-following capabilities. Therefore, prompt engineering and output validation become critical components of the experimental pipeline.

You may choose one or both of the following approaches:

### Option A: Local Open-Source LLMs

Use a lightweight or quantized open-source LLM locally without performing additional training or fine-tuning. Possible examples include:

* Gemma
* Llama
* Mistral
* Phi
* Qwen

You may run these models using frameworks such as Hugging Face Transformers, llama.cpp, Ollama, or vLLM. This option is recommended for those who are interested in local deployment, hardware efficiency, or fully offline experimentation.

### Option B: Cloud API-Based Inference (Recommended)

Use the **Google Gemini API** to access Gemma 3 or Gemma 4 through Google AI Studio. This option simplifies deployment and typically provides stronger instruction-following performance.

> **Important:** Free-tier usage is limited to approximately **1,500 requests per day per model**. Plan your experiments carefully before running inference on the full test set. You may refer to `Google AI Studio` for more information.

### Prompt Engineering and Analysis

Since the LLM is not fine-tuned specifically for NLI, the prompt acts as the primary mechanism for guiding the model’s reasoning process. You must design a clear and robust prompt that:

* Explains the NLI task
* Defines the three target labels (Entailment, Neutral, Contradiction)
* Specifies the expected output format
* Minimizes ambiguous or verbose responses

> **Note:** You are **strictly forbidden** from including any examples from the **test set** inside your prompt.

However, you may:

* Use observations from the training set
* Use few-shot examples sampled only from the training split
* Incorporate heuristics discovered during exploratory data analysis

### Output Formatting

You must instruct the LLM to return outputs in a strict and parsable format. Recommended formats include:

\```text

Entailment

\```

or

\```json

{"label": "Neutral"}

\```

The simpler the output format, the easier it will be to parse and evaluate automatically.

### Inference on the Test Set

Run inference on the same test split used during BERT evaluation.

> **Note:** Due to the computational requirements of LLMs and/or free API rate limitations, you may optionally perform inference on **the first subset of 1,000 test samples** instead of using the entire test set.

Because LLMs are non-deterministic and may fail to follow instructions consistently, you must implement a robust parsing and validation pipeline. Your implementation should properly handle:

* **Valid Predictions:** Outputs that can be mapped correctly to one of the three labels (Entailment, Neutral, Contradiction).
* **Invalid Predictions:** Cases where the model produces ambiguous outputs, hallucinates additional explanations, returns malformed JSON, generates labels outside the allowed classes, or ignores formatting instructions. You should explicitly track and report invalid outputs during evaluation.

---
### Task 3: Your Work Starts Here

### Approach (Option A: local open-source LLMs through Ollama)

- **Models:** `qwen2.5:7b` and `llama3.2:3b` (4-bit quantized Ollama builds), no fine-tuning. Two sizes show whether a smaller
  model follows the output format less reliably.
- **Decoding:** temperature 0, fixed seed, at most 16 new tokens.
- **Prompts:** a zero-shot prompt and a few-shot prompt with one example per label, sampled **only from the training split**.
- **Output format:** a single word (`Entailment`, `Neutral` or `Contradiction`). Raw answers are stored unchanged, so parsing
  can be changed later without calling the models again.

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"
LLM_MODELS = ["qwen2.5:7b", "llama3.2:3b"]

def ollama_up():
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).ok
    except requests.exceptions.RequestException:
        return False

if not ollama_up():
    if shutil.which("ollama") is None:   # Kaggle / Colab: install the Linux build
        subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null 2>&1", shell=True)
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
assert ollama_up(), "Ollama server did not start"

for m in LLM_MODELS:
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    print(m, "->", "ok" if r.returncode == 0 else r.stderr[-300:])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

In [ ]:
SYSTEM_PROMPT = """You are an expert annotator for natural language inference (NLI).
Given a premise and a hypothesis, decide how they relate:
- Entailment: the hypothesis is definitely true if the premise is true.
- Contradiction: the hypothesis is definitely false if the premise is true.
- Neutral: the premise neither confirms nor rules out the hypothesis.
Reply with exactly one word: Entailment, Neutral, or Contradiction. Do not explain."""

def user_message(premise, hypothesis):
    return f"Premise: {premise}\nHypothesis: {hypothesis}\nAnswer:"

# Few-shot examples: one per label, sampled from the TRAINING split only
shot_pool = ds["train"].shuffle(seed=SEED).select(range(2000))
few_shot = []
for lab in range(3):
    ex = next(e for e in shot_pool if e["label"] == lab)
    few_shot.append((ex["premise"], ex["hypothesis"], LABELS[lab]))
for p, h, l in few_shot:
    print(f"[{l}] P: {p} | H: {h}")

def build_messages(premise, hypothesis, mode):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    if mode == "few_shot":
        for p, h, l in few_shot:
            msgs += [{"role": "user", "content": user_message(p, h)}, {"role": "assistant", "content": l}]
    msgs.append({"role": "user", "content": user_message(premise, hypothesis)})
    return msgs

In [ ]:
def ask(model, messages):
    t = time.perf_counter()
    r = requests.post(f"{OLLAMA_URL}/api/chat", timeout=600, json={
        "model": model, "messages": messages, "stream": False,
        "options": {"temperature": 0, "seed": SEED, "num_predict": 16}})
    r.raise_for_status()
    return r.json()["message"]["content"], time.perf_counter() - t

def run_llm(model, mode):
    """Query the model on the comparison subset; raw answers are cached in a CSV so runs can resume."""
    path = os.path.join(OUT_DIR, f"llm_raw_{model.replace(':', '_').replace('.', '_')}_{mode}.csv")
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=["idx", "raw_output", "latency_s"])
    seen = set(done["idx"])
    ask(model, build_messages("A man is sleeping.", "A person rests.", mode))   # warm-up: loads the model
    rows = []
    for r in compare_df.itertuples():
        if r.idx in seen:
            continue
        out, sec = ask(model, build_messages(r.premise, r.hypothesis, mode))
        rows.append({"idx": r.idx, "raw_output": out, "latency_s": sec})
        if len(rows) % 100 == 0:
            done = pd.concat([done, pd.DataFrame(rows)]); rows = []
            done.to_csv(path, index=False)
            print(f"  {model} {mode}: {len(done)}/{len(compare_df)}")
    done = pd.concat([done, pd.DataFrame(rows)]).sort_values("idx").reset_index(drop=True)
    done.to_csv(path, index=False)
    return done

llm_raw = {}
for model in LLM_MODELS:
    for mode in ["zero_shot", "few_shot"]:
        t0 = time.time()
        llm_raw[(model, mode)] = run_llm(model, mode)
        print(f"{model} {mode}: {len(llm_raw[(model, mode)])} answers in {(time.time() - t0) / 60:.1f} min")

### Parsing and validation

Each raw answer is mapped to a label or marked **invalid**:

| Rule | Meaning | Result |
|---|---|---|
| `exact` | the cleaned answer is exactly one label name | valid |
| `json` | valid JSON with a `label` field holding one label name | valid |
| `contained` | extra words, but exactly one label name appears | valid |
| `multiple_labels` | more than one label name appears (e.g. hedging) | invalid |
| `no_label` | no label name at all (empty, other word, explanation only) | invalid |

Invalid answers are kept in the evaluation and counted as wrong.

In [ ]:
LABEL_WORDS = {l.lower(): i for i, l in enumerate(LABELS)}

def parse_label(text):
    t = str(text).strip()
    try:
        obj = json.loads(t)
        if isinstance(obj, dict) and str(obj.get("label", "")).strip().lower() in LABEL_WORDS:
            return LABEL_WORDS[str(obj["label"]).strip().lower()], "json"
    except (json.JSONDecodeError, TypeError):
        pass
    tokens = re.findall(r"[a-z]+", t.lower())
    found = {LABEL_WORDS[w] for w in tokens if w in LABEL_WORDS}
    if len(found) > 1:
        return -1, "multiple_labels"
    if not found:
        return -1, "no_label"
    return found.pop(), ("exact" if len(tokens) == 1 else "contained")

# quick self-test
for s in ["Entailment", " neutral.", '{"label": "Contradiction"}', "The answer is Neutral",
          "Entailment or Neutral", "I cannot tell", ""]:
    print(repr(s), "->", parse_label(s))

parsed = {}
for key, raw_df in llm_raw.items():
    d = raw_df.merge(compare_df[["idx", "gold"]], on="idx")
    d[["pred", "rule"]] = d["raw_output"].apply(lambda s: pd.Series(parse_label(s)))
    parsed[key] = d
    d.to_csv(os.path.join(OUT_DIR, f"llm_parsed_{key[0].replace(':', '_').replace('.', '_')}_{key[1]}.csv"), index=False)

rule_table = pd.DataFrame({f"{m} | {mode}": d["rule"].value_counts() for (m, mode), d in parsed.items()}).fillna(0).astype(int)
display(rule_table)
for (m, mode), d in parsed.items():
    bad = d[d.pred == -1].head(3)
    for r in bad.itertuples():
        print(f"[{m} {mode}] invalid ({r.rule}): {r.raw_output!r}")

### Task 3: Your Work Ends Here
---

## Task 4: Visualization and Comparison

### 4.1 LLM Metric Calculation

Unlike BERT classifiers, LLMs may generate invalid or unparsable responses. Therefore, additional evaluation metrics are required. You must calculate the following:

**Overall Accuracy:**


$$\frac{\text{Correct Predictions}}{\text{Total Samples}}$$


*(This metric penalizes invalid responses.)*

**Invalid Rate:**


$$\frac{\text{Invalid Predictions}}{\text{Total Samples}}$$


*(This measures how often the LLM fails to follow instructions correctly.)*

**Adjusted Accuracy:**


$$\frac{\text{Correct Predictions}}{\text{Valid Predictions Only}}$$


*(This metric evaluates the model only on successfully parsed predictions.)*

**Macro F1-Score:**
Compute the macro-averaged F1-score over the entire test subset. You must NOT drop or exclude invalid, unparsable, or missing predictions from your dataset. Any such output must be strictly treated as an incorrect prediction (misclassification) during your metric calculations.

### 4.2 Visual Analysis

Generate the following visualizations:

**Side-by-Side Confusion Matrices**

> Use the same 1000 test split and evaluate your fine-tuned BERT model and LLM pipeline.

Display confusion matrices for:

1. Fine-tuned BERT
2. LLM-based inference

You may include an additional category such as `"Invalid"` or `"N/A"` to visualize LLM formatting failures or parsing issues.

**Performance Comparison Bar Chart**
Create a bar chart comparing:

* Accuracy
* Adjusted Accuracy
* Macro F1-Score
* Invalid Rate

across both approaches. You are encouraged to include clear labels, legends, and interpretations.

---
### Task 4: Your Work Starts Here

In [ ]:
def nli_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    valid = y_pred != -1
    correct = y_pred == y_true
    return {
        "accuracy": float(correct.mean()),
        "invalid_rate": float(1 - valid.mean()),
        "adjusted_accuracy": float(correct[valid].mean()) if valid.any() else 0.0,
        # invalid predictions (-1) are kept and can never match a gold label, so they count as errors
        "macro_f1": float(f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
    }

systems = {"BERT (fine-tuned)": (compare_df["gold"], compare_df["bert_pred"])}
latency = {"BERT (fine-tuned)": bert_test["single_ms_per_example"]}
for (m, mode), d in parsed.items():
    name = f"{m} {mode.replace('_', '-')}"
    systems[name] = (d["gold"], d["pred"])
    latency[name] = 1000 * float(d["latency_s"].median())

summary = pd.DataFrame({k: nli_metrics(*v) for k, v in systems.items()}).T
summary["median_ms_per_example"] = pd.Series(latency)
summary.index.name = f"system (first {len(compare_df)} test samples)"
display(summary.round(4))
summary.to_csv(os.path.join(OUT_DIR, "comparison_metrics.csv"))

In [ ]:
# Side-by-side confusion matrices; LLMs get an extra "Invalid" column
names = list(systems)
cols = LABELS + ["Invalid"]
fig, axes = plt.subplots(1, len(names), figsize=(5.2 * len(names), 4.6))
for ax, name in zip(np.atleast_1d(axes), names):
    yt, yp = systems[name]
    m = pd.crosstab(pd.Series(yt, name="true").map(dict(enumerate(LABELS))),
                    pd.Series(yp, name="pred").map({**dict(enumerate(LABELS)), -1: "Invalid"}))
    m = m.reindex(index=LABELS, columns=cols, fill_value=0)
    if name.startswith("BERT"):
        m = m[LABELS]
    sns.heatmap(m, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(name, fontsize=10); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); savefig("t4_confusion_matrices.png"); plt.show()

# Grouped bar chart of the four metrics
plot_df = summary[["accuracy", "adjusted_accuracy", "macro_f1", "invalid_rate"]]
ax = plot_df.T.plot.bar(figsize=(12, 5), rot=0, width=.8)
ax.set_ylim(0, 1); ax.set_title(f"BERT vs. LLMs on the first {len(compare_df)} SNLI test samples")
ax.legend(fontsize=8, loc="upper right")
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=7)
plt.tight_layout(); savefig("t4_metric_comparison.png"); plt.show()

In [ ]:
# Run record, used for the discussion and the README
llm_sizes = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
run_record = {
    "gpu": GPU_NAME, "torch": torch.__version__, "smoke_test": SMOKE,
    "bert": bert_test, "comparison": summary.round(4).to_dict(orient="index"),
    "ollama_models": llm_sizes,
}
json.dump(run_record, open(os.path.join(OUT_DIR, "run_record.json"), "w"), indent=1, default=str)
print(json.dumps(run_record, indent=1, default=str)[:3000])

### Task 4: Your Work Ends Here
---

## Task 5: Comparative Report and Discussion

Write a concise but analytical discussion comparing the fine-tuned BERT model and the LLM-based approach. Your discussion should address the following aspects:

### 5.1 Performance vs. Reliability

* Which model achieved higher accuracy?
* Did the LLM suffer from formatting inconsistencies?
* Was the invalid prediction rate significant?
* Did BERT’s deterministic behavior provide advantages for production usage?
* Discuss the trade-off between flexibility and reliability.

### 5.2 Error Analysis

Using the confusion matrices:

* Identify which labels were most difficult for each model.
* Analyze whether these 2 models struggled particularly with Neutral, Contradiction, or Entailment.
* Discuss possible reasons behind these behaviors.

### 5.3 Cost and Efficiency Analysis

Compare the approaches in terms of:

* Hardware requirements
* Inference latency
* API costs
* Memory consumption
* Ease of deployment
* Scalability
* Practical trade-offs between fine-tuned task-specific models and general-purpose LLMs.

Consider the following scenario:

> A production RAG (Retrieval-Augmented Generation) system needs to verify whether a generated answer is logically supported by retrieved documents.

Which approach would you deploy?

* Fine-tuned BERT classifier?
* Prompted LLM?
* Hybrid pipeline?

Justify your decision in terms of Reliability, Cost-efficiency, Latency, Robustness, and Maintainability.

---
### Task 5: Your Work Starts Here

*To be written after reviewing the results above. Use the numbers in `comparison_metrics.csv` and `run_record.json`.*

#### 5.1 Performance vs. reliability
-

#### 5.2 Error analysis
-

#### 5.3 Cost and efficiency, and the RAG verification scenario
-

### Task 3: Your Work Ends Here
---

## Additional Notes

* There is no strict format for outputs or results. You are encouraged to do your best.
* Organize your notebook clearly using:
  * **Markdown cells** for explanations, observations, and comments
  * **Code cells** for implementation and experiments
* Explain important implementation decisions.
* Include sufficient comments throughout your code.
* Do **not** include API keys or credentials in your submission.
* Ensure your experiments are reproducible.

> **Important:**
> The sections between two horizontal lines (e.g., “Task 1: Your Work Starts/Ends Here”) are your work areas. Within these sections, you can add as many new cells as needed between the lines to complete your solution. Please ensure that you follow the existing notebook structure and do not delete or modify any pre-existing content outside these marked regions.

## Submission Guidelines

* You must submit **only the notebook file** compressed as a `.zip` archive. Model weights or checkpoints are not required.
* The notebook filename must follow the format: `<studentID>.ipynb` (e.g., `20212223.ipynb`).
* The **ZIP file name must also be the same as your student ID** (e.g., `20212223.zip`).
* Before submission, make sure that:
    * All notebook cells have been executed
    * Outputs, plots, and visualizations are visible inside the notebook
    * The notebook is saved properly with outputs included



> **Important:**
> This notebook serves as your final report file. Therefore, submissions with missing outputs, missing visualizations, or improperly named files may receive a score reduction.